In [ ]:
import os
import logging
from dotenv import load_dotenv
from logging.handlers import RotatingFileHandler
import json
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, answer_similarity, answer_relevancy
#from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import nest_asyncio
from langchain_upstage import ChatUpstage
from langchain_upstage import UpstageEmbeddings

In [ ]:
# 환경 변수 로드
load_dotenv()
#print(os.getenv("OPENAI_API_KEY"))
print(os.getenv("UPSTAGE_API_KEY"))

# 로거 설정
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
console_handler = logging.StreamHandler()
logger.addHandler(console_handler)

## 요약 3개 생성하기

In [ ]:
import pandas as pd
from tqdm import tqdm
import os
from dotenv import load_dotenv

# 1. API 키 로드
load_dotenv()

client = ChatUpstage(
    model="solar-pro",
    temperature=0.5
)

# ---------------------------------------------------------
# [함수] 3가지 버전의 요약 생성
# ---------------------------------------------------------
def generate_three_summaries(original_answer, question):
    summaries = []
    
    system_message = """
    You are a wise and compassionate **Christian** counselor chatbot. 
    Your goal is to share the love of Jesus through gentle, conversational interactions.
    
    **Instructions:**
    1.  **Summarize:** The user provides a 'Question' and a 'Long Theological Answer'. You must rewrite the answer into a concise, conversational response.
    2.  **Tone:** Speak naturally like a caring friend or pastor, not like a textbook or a search engine. Be empathetic and warm.
    3.  **Length:** Keep it relatively short (2-4 sentences usually, unless the topic requires more nuance), suitable for a chat interface.
    4.  **Content:** Base your answer STRICTLY on the provided 'Long Theological Answer'. Do not invent new theology, but you can phrase it more simply.
    5.  **Style:** Avoid heavy theological jargon where possible. If the topic is sensitive, show understanding.
    """
    
    for i in range(3):
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": f"Question: {question}\n\nOriginal Answer: {original_answer}\n\nSummarized Answer {i+1}:"}
                ],
                temperature=0.7 + (i * 0.1) 
            )
            summaries.append(response.choices[0].message.content.strip())
        except Exception as e:
            # 에러 발생 시 빈 문자열 대신 에러 메시지를 남기거나 비워둡니다.
            print(f"⚠️ Error generating summary {i+1}: {e}")
            summaries.append("") # 실패하면 빈칸 저장
    
    return summaries

# ---------------------------------------------------------
# [실행] 데이터셋에 적용 (중간 저장 기능 포함)
# ---------------------------------------------------------
input_file = "GotQuestions_raw_Ara_2025-05-01.xlsx"
df = pd.read_excel(input_file)

# 결과를 담을 리스트
summary_results = []

# 최종 저장 파일명
output_file = "GQ_3answers_2175.xlsx"
# 중간 백업 파일명
backup_file = "3Ans_GQ_backup.xlsx"

print(f"🚀 총 {len(df)}개 데이터에 대해 요약 생성 시작...")

# enumerate를 사용하여 몇 번째인지 카운트
for i, (index, row) in enumerate(tqdm(df.iterrows(), total=df.shape[0])):
    try:
        question = row['Question_ENG']
        original = row['Answer_ENG']
        
        # 3가지 요약 생성
        three_summaries = generate_three_summaries(original, question)
        
        summary_results.append({
            "Question": question,
            "Original_Answer": original,
            "Summary1": three_summaries[0],
            "Summary2": three_summaries[1],
            "Summary3": three_summaries[2]
        })
        
        # ★안전장치★: 10개 할 때마다 파일로 저장 (백업)
        # 컴퓨터가 꺼지거나 API 에러로 멈춰도 이 파일은 남음
        if (i + 1) % 10 == 0:
            pd.DataFrame(summary_results).to_excel(backup_file, index=False)
            
    except Exception as e:
        print(f"❌ Row {i} Error: {e}")
        # 행 전체 에러가 나도 멈추지 않고 다음 행으로 넘어가기
        continue

# 반복문이 다 끝나면 최종 파일 저장
result_df = pd.DataFrame(summary_results)
result_df.to_excel(output_file, index=False)

print(f"✅ 모든 작업 완료! 최종 파일: {output_file}")

## Ragas로 Bast 답변 계산

In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, answer_similarity, answer_relevancy
import nest_asyncio
import time

# 1. 환경 설정 및 모델 준비
nest_asyncio.apply()

model = ChatUpstage(model="solar-pro", temperature=0.5)
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

#model = ChatOpenAI(model="gpt-4o")
#embeddings = OpenAIEmbeddings()

# 2. 파일 로드
file_summary = "GQ_3answers_2175.xlsx"
file_original = "GotQuestions_raw_Ara_2025-05-01.xlsx"
output_file = "GQ_3Summaries_Ragas_Progress.xlsx" # 중간 저장 파일명

print(f"📂 파일 로딩 중...")
df_summary = pd.read_excel(file_summary)
df_original = pd.read_excel(file_original)

# 3. 데이터 병합 (이전과 동일 로직)
def get_question_col(df):
    for col in ['Question', 'Question_ENG']:
        if col in df.columns:
            return col
    return df.columns[0]

q_col_sum = get_question_col(df_summary)
q_col_orig = get_question_col(df_original)

df_merged = pd.merge(
    df_summary, 
    df_original[[q_col_orig, 'Answer_ENG']], 
    left_on=q_col_sum, 
    right_on=q_col_orig, 
    how='left'
)

ground_truth_col = 'Answer_ENG'
final_q_col = q_col_sum
summary_cols = ['Summary1', 'Summary2', 'Summary3']

# 4. 배치 처리 설정
BATCH_SIZE = 10  # 10개씩 처리
total_rows = len(df_merged)
processed_results = [] # 결과를 모을 리스트

print(f"🚀 총 {total_rows}개의 데이터를 {BATCH_SIZE}개씩 나누어 평가합니다.")

# 가중치 설정
W_REL, W_COR, W_SIM = 0.5, 0.22, 0.28

def pick_best_weighted(row):
    """행별로 점수를 계산해 Best를 뽑는 함수"""
    scores = []
    answers = []
    
    for i in range(1, 4):
        # 점수가 계산되지 않았거나 에러인 경우 0점 처리
        s_rel = row.get(f'Summary{i}_Relevancy', 0)
        s_cor = row.get(f'Summary{i}_Correctness', 0)
        s_sim = row.get(f'Summary{i}_Similarity', 0)
        
        # NaN 체크
        if pd.isna(s_rel): s_rel = 0
        if pd.isna(s_cor): s_cor = 0
        if pd.isna(s_sim): s_sim = 0
        
        weighted_score = (s_rel * W_REL) + (s_cor * W_COR) + (s_sim * W_SIM)
        scores.append(weighted_score)
        answers.append(row.get(f'Summary{i}', ""))
    
    max_score = max(scores)
    max_index = scores.index(max_score)
    
    return pd.Series({
        'Best_Summary': answers[max_index], 
        'Weighted_Score': max_score, 
        'Best_Source_Num': f"Summary {max_index+1}"
    })

# 5. 메인 루프 (배치 단위 실행)
for start_idx in range(0, total_rows, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total_rows)
    print(f"\n🔄 Processing Batch: {start_idx} ~ {end_idx} (Total: {total_rows})")
    
    # 현재 배치 데이터 슬라이싱
    batch_df = df_merged.iloc[start_idx:end_idx].copy()
    
    # 필수 데이터 리스트 변환 (NaN 처리 포함)
    questions = batch_df[final_q_col].fillna("").astype(str).tolist()
    ground_truths = batch_df[ground_truth_col].fillna("").astype(str).tolist()
    contexts = [[gt] for gt in ground_truths]
    
    # 현재 배치의 결과를 담을 임시 데이터프레임 (원본 복사)
    current_batch_result = batch_df.copy()
    
    try:
        # Summary 1, 2, 3 각각 평가
        for i, col in enumerate(summary_cols):
            num = i + 1
            answers = batch_df[col].fillna("").astype(str).tolist()
            
            # Ragas 데이터셋 생성
            data_dict = {
                "question": questions,
                "ground_truth": ground_truths,
                "answer": answers,
                "contexts": contexts
            }
            dataset = Dataset.from_dict(data_dict)
            
            print(f"   ...Evaluating {col} ({len(dataset)} items)")
            
            # 평가 수행
            results = evaluate(
                dataset=dataset,
                metrics=[answer_correctness, answer_similarity, answer_relevancy],
                llm=model,
                embeddings=embeddings,
                raise_exceptions=False # 에러 발생 시 멈추지 않고 NaN 반환 시도
            )
            
            # 결과 DataFrame으로 변환
            res_df = results.to_pandas()
            
            # 결과를 현재 배치 DataFrame에 컬럼으로 추가
            current_batch_result[f'Summary{num}_Correctness'] = res_df['answer_correctness'].values
            current_batch_result[f'Summary{num}_Similarity'] = res_df['answer_similarity'].values
            current_batch_result[f'Summary{num}_Relevancy'] = res_df['answer_relevancy'].values

        # Best Pick 계산
        best_pick_df = current_batch_result.apply(pick_best_weighted, axis=1)
        current_batch_result = pd.concat([current_batch_result, best_pick_df], axis=1)
        
        # 결과 리스트에 추가
        processed_results.append(current_batch_result)
        
        # --- 중간 저장 ----
        # 지금까지 처리된 모든 결과 합치기
        all_results_so_far = pd.concat(processed_results, axis=0)
        
        # 컬럼 순서 정리 (보기 좋게)
        cols_order = [final_q_col, ground_truth_col] + summary_cols
        metrics_cols = []
        for i in range(1, 4):
            metrics_cols.extend([f'Summary{i}_Correctness', f'Summary{i}_Similarity', f'Summary{i}_Relevancy'])
        final_cols = cols_order + metrics_cols + ['Best_Source_Num', 'Weighted_Score', 'Best_Summary']
        
        # 존재하는 컬럼만 선택하여 저장
        existing_cols = [c for c in final_cols if c in all_results_so_far.columns]
        # 나머지 기타 컬럼들 뒤에 붙이기
        remaining_cols = [c for c in all_results_so_far.columns if c not in existing_cols]
        
        save_df = all_results_so_far[existing_cols + remaining_cols]
        save_df.to_excel(output_file, index=False)
        
        print(f"   💾 중간 저장 완료: {end_idx}행까지 저장됨 -> {output_file}")

    except Exception as e:
        print(f"❌ [CRITICAL ERROR] Batch {start_idx}~{end_idx} 처리 중 오류 발생!")
        print(f"Error Message: {e}")
        print("🚨 현재까지 저장된 파일을 유지하고 멈춥니다. API 키를 확인하거나 나중에 다시 시도하세요.")
        break # 루프 중단

print("\n✨ 모든 작업 종료.")
if processed_results:
    print(f"최종 파일은 '{output_file}'에 저장되어 있습니다.")

## 중간에 멈췄다면 멈춘 부분부터 실행할 수 있도록 하기

In [ ]:
import sys
import types
from unittest.mock import MagicMock

# 1. 기존에 잘못 설정된 가짜 모듈이 있다면 제거 (청소)
if "google.generativeai" in sys.modules:
    del sys.modules["google.generativeai"]
if "google.ai.generativelanguage" in sys.modules:
    del sys.modules["google.ai.generativelanguage"]

# 2. 'google.generativeai'를 위한 정교한 가짜 모듈 생성
# 단순히 MagicMock()만 쓰면 __spec__ 에러가 나므로, 실제 모듈 객체(ModuleType)를 사용
mock_genai = types.ModuleType("google.generativeai")
mock_genai.__spec__ = MagicMock()       # __spec__ 속성 설정 (에러 방지 핵심!)
mock_genai.__path__ = []                # 패키지처럼 보이게 설정

# 3. sys.modules에 주입 (이제 파이썬은 이 모듈이 정상적으로 설치됐다고 착각합니다)
sys.modules["google.generativeai"] = mock_genai
sys.modules["google.ai.generativelanguage"] = MagicMock()

import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, answer_similarity, answer_relevancy
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import nest_asyncio
import os
from dotenv import load_dotenv
import logging


# 1. 환경 설정
nest_asyncio.apply()
# 경고 메시지 숨기기
logging.getLogger("ragas").setLevel(logging.ERROR)
logging.getLogger("langchain").setLevel(logging.ERROR)

model = ChatUpstage(model="solar-pro", temperature=0.5)
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
#model = ChatOpenAI(model="gpt-4o")
#embeddings = OpenAIEmbeddings()

# 2. 파일 로드
file_summary = "3Ans_GQ.xlsx"
file_original = "GotQuestions_raw_Ara_2025-05-01.xlsx"
output_file = "GQ_3Summaries_Ragas_Progress.xlsx" # 중간 저장 파일명

print(f"📂 원본 데이터 로딩 중...")
df_summary = pd.read_excel(file_summary)
df_original = pd.read_excel(file_original)

# 데이터 병합 (이전과 동일)
def get_question_col(df):
    for col in ['Question', 'Question_ENG']:
        if col in df.columns:
            return col
    return df.columns[0]

q_col_sum = get_question_col(df_summary)
q_col_orig = get_question_col(df_original)

df_merged = pd.merge(
    df_summary, 
    df_original[[q_col_orig, 'Answer_ENG']], 
    left_on=q_col_sum, 
    right_on=q_col_orig, 
    how='left'
)

ground_truth_col = 'Answer_ENG'
final_q_col = q_col_sum
summary_cols = ['Summary1', 'Summary2', 'Summary3']

# 3. 이어하기 설정 (Resume Logic)
processed_results = []
start_idx_resume = 0

if os.path.exists(output_file):
    print(f"🔄 기존 진행 파일 발견: {output_file}")
    df_existing = pd.read_excel(output_file)
    start_idx_resume = len(df_existing) # 이미 처리된 행의 개수
    processed_results.append(df_existing) # 기존 결과 메모리에 로드
    print(f"   ▶ {start_idx_resume}개 데이터가 이미 처리되었습니다. {start_idx_resume}번부터 이어서 시작합니다.")
else:
    print("   ▶ 기존 파일이 없습니다. 처음(0번)부터 시작합니다.")

# 4. 배치 처리 설정
BATCH_SIZE = 10
total_rows = len(df_merged)
W_REL, W_COR, W_SIM = 0.5, 0.22, 0.28 # 가중치

def pick_best_weighted(row):
    scores = []
    answers = []
    for i in range(1, 4):
        s_rel = row.get(f'Summary{i}_Relevancy', 0)
        s_cor = row.get(f'Summary{i}_Correctness', 0)
        s_sim = row.get(f'Summary{i}_Similarity', 0)
        if pd.isna(s_rel): s_rel = 0
        if pd.isna(s_cor): s_cor = 0
        if pd.isna(s_sim): s_sim = 0
        
        weighted_score = (s_rel * W_REL) + (s_cor * W_COR) + (s_sim * W_SIM)
        scores.append(weighted_score)
        answers.append(row.get(f'Summary{i}', ""))
    
    max_score = max(scores)
    max_index = scores.index(max_score)
    return pd.Series({
        'Best_Summary': answers[max_index], 
        'Weighted_Score': max_score, 
        'Best_Source_Num': f"Summary {max_index+1}"
    })

# 5. 메인 루프 (중단된 지점부터 시작)
# range 시작점을 start_idx_resume로 설정하여 건너뛰기 구현
# 주의: BATCH_SIZE 단위로 딱 떨어지지 않을 수 있으므로 조정
current_start = (start_idx_resume // BATCH_SIZE) * BATCH_SIZE 
if start_idx_resume % BATCH_SIZE != 0:
    # 혹시 중간에 애매하게 끊겼다면, 안전하게 그 배치 처음부터 다시 하도록 설정
    print(f"   ⚠️ 배치가 중간에 끊겼습니다. {current_start}번부터 다시 처리합니다.")
else:
    current_start = start_idx_resume

for start_idx in range(current_start, total_rows, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total_rows)
    print(f"\n🔄 Processing Batch: {start_idx} ~ {end_idx} (Total: {total_rows})")
    
    batch_df = df_merged.iloc[start_idx:end_idx].copy()
    
    questions = batch_df[final_q_col].fillna("").astype(str).tolist()
    ground_truths = batch_df[ground_truth_col].fillna("").astype(str).tolist()
    contexts = [[gt] for gt in ground_truths]
    
    current_batch_result = batch_df.copy()
    
    try:
        for i, col in enumerate(summary_cols):
            num = i + 1
            answers = batch_df[col].fillna("").astype(str).tolist()
            
            data_dict = {
                "question": questions,
                "ground_truth": ground_truths,
                "answer": answers,
                "contexts": contexts
            }
            dataset = Dataset.from_dict(data_dict)
            
            print(f"   ...Evaluating {col}")
            results = evaluate(
                dataset=dataset,
                metrics=[answer_correctness, answer_similarity, answer_relevancy],
                llm=model,
                embeddings=embeddings,
                raise_exceptions=False
            )
            res_df = results.to_pandas()
            current_batch_result[f'Summary{num}_Correctness'] = res_df['answer_correctness'].values
            current_batch_result[f'Summary{num}_Similarity'] = res_df['answer_similarity'].values
            current_batch_result[f'Summary{num}_Relevancy'] = res_df['answer_relevancy'].values

        best_pick_df = current_batch_result.apply(pick_best_weighted, axis=1)
        current_batch_result = pd.concat([current_batch_result, best_pick_df], axis=1)
        
        # 결과 리스트에 추가 (주의: 이미 있는 데이터면 덮어쓰거나 건너뛰어야 함)
        # 여기서는 매번 전체를 다시 concat해서 저장하는 방식 (가장 안전)
        if len(processed_results) > 0 and start_idx < len(pd.concat(processed_results)):
             # 만약 재시작으로 인해 겹치는 부분이 있다면, 기존 리스트에서 该 부분을 제거하고 새거 추가
             # 복잡함을 피하기 위해, 그냥 기존 파일 읽은거 + 새로 한거 합치기
             pass
        else:
             pass 

        # *** 중요: 이어쓰기 저장 로직 ***
        # 매 배치마다 파일 끝에 추가(append)하는 것이 효율적이지만, 엑셀은 append가 어려움
        # 따라서, '기존에 읽어온 df_existing'이 있다면 그것과 '새로 한 batch'를 합쳐서 저장
        
        if 'df_existing' in locals() and start_idx == current_start:
             # 첫 루프에서는 기존 것과 합치지 않고, processed_results를 초기화
             # 위에서 이미 append 했으므로 pass
             pass
        else:
             processed_results.append(current_batch_result)

        # 전체 병합 및 저장
        all_results_so_far = pd.concat(processed_results, axis=0)
        # 중복 제거 (혹시 재시작 시점에 겹친 행이 있다면 제거)
        all_results_so_far = all_results_so_far.drop_duplicates(subset=[final_q_col], keep='last')
        
        # 컬럼 순서 정리
        cols_order = [final_q_col, ground_truth_col] + summary_cols
        metrics_cols = []
        for i in range(1, 4):
            metrics_cols.extend([f'Summary{i}_Correctness', f'Summary{i}_Similarity', f'Summary{i}_Relevancy'])
        final_cols = cols_order + metrics_cols + ['Best_Source_Num', 'Weighted_Score', 'Best_Summary']
        existing_cols = [c for c in final_cols if c in all_results_so_far.columns]
        remaining_cols = [c for c in all_results_so_far.columns if c not in existing_cols]
        
        save_df = all_results_so_far[existing_cols + remaining_cols]
        save_df.to_excel(output_file, index=False)
        
        print(f"   💾 저장 완료: {len(save_df)}행 저장됨.")
        
        print("   💤 5초 대기...")
        time.sleep(5)

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
        break

print("✨ 작업 종료.")